## DSPy Prompts to Assess Bias in Questions

### Setting up DSPy

In [ ]:
# Import libraries
import dspy
from dspy.teleprompt import BootstrapFewShot
from dspy.teleprompt import BootstrapFewShotWithRandomSearch
from dspy.evaluate.evaluate import Evaluate
import json
import random

import os
from dotenv import load_dotenv
load_dotenv()
open_ai_api_key = os.getenv("OPENAI_API_KEY")


c:\Users\Cassandra\miniconda3\envs\coalesce\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Set up the LM (https://dspy-docs.vercel.app/api/language_model_clients/OpenAI)
gpt3_turbo = dspy.OpenAI(model='gpt-3.5-turbo', max_tokens=500, api_key=open_ai_api_key)  
dspy.configure(lm=gpt3_turbo)

### Create AssessBias Signature

In [3]:
# Create a class-based DSPy Signature to assess readability of a question

input_description = """The question to classify. The input will be a JSON with the following structure:
        {
            "response_format": "open" or "closed",
            "description": string,
            "main_text": string,
            "response_categories": empty list or list of JSONs with an "id" and "text" field
        }"""

# output_description = """Whether a question has bias or not. Only output one of the following strings: "true" or "false". Don't include any other information in the output.""" 
output_description = """Whether a question has bias or not. Output "true" if the question is biased, and "false" if the question is not biased.""" 

class AssessBias(dspy.Signature):
    """Assess whether a question is biased or not. Keep in mind the following forms of bias:
        (1) Leading questions, which are phrased in a way that suggests a particular answer is more desirable or correct. They tend to have subjective adjectives, or context-laden words that frame the question in a positive or negative light.
        (2) Making assumptions about respondents' behaviors, attitudes, or goals. These questions tend to guess information instead of asking for it.
        (3) Double-barreled questions, which ask about two or more things simultaneously.
        (4) Emotionally loaded language, which has emotionally loaded terms or phrases that imply judgment or assume a particular stance. Words that carry strong positive or negative connotations can influence respondents' emotions and responses."""

    # question = dspy.InputField(desc="The question to assess.")
    question = dspy.InputField(desc=input_description)
    bias = dspy.OutputField(desc=output_description)

#### Test AssessBias Signature

In [95]:
test_inputs = [
    {
        "response_format": "open",
        "description": "",
        "main_text": "Did the tornado sound like a freight train?",
        "response_categories": []
    },
    {
        "response_format": "open",
        "description": "",
        "main_text": "What did the tornado sound like?",
        "response_categories": []
    },
    {
        "response_format": "open",
        "description": "",
        "main_text": "Is the current bus timing and frequency helpful to you?",
        "response_categories": []
    }
]

In [96]:
# test out AssessBias

rand_int = random.randint(1, 100)

bias = dspy.ChainOfThought(AssessBias)

# running the predictor
for test_input in test_inputs:
    # stringify the input
    test_input_str = json.dumps(test_input)
    # test_input_str = test_input["main_text"]
    result = bias(question=test_input_str, config=dict(temperature=0.7+0.0001*rand_int))
    rationale = result.rationale
    bias_output = result.bias
    print(f"The bias of the question '{test_input["main_text"]}' is {bias_output}.") 
    print(f"Rationale: {rationale}")
    print()
    print()

The bias of the question 'Did the tornado sound like a freight train?' is true.
Rationale: produce the bias. We can see that this question is making an assumption about the sound of a tornado, comparing it to a freight train. This assumption may influence the respondent's answer.


The bias of the question 'What did the tornado sound like?' is false.
Rationale: produce the bias. We can see that this question is asking for a specific detail about the tornado, which does not contain any leading language, assumptions, double-barreled elements, or emotionally loaded terms.


The bias of the question 'Is the current bus timing and frequency helpful to you?' is true.
Rationale: produce the bias. We see that the question is asking for the respondent's opinion on the bus timing and frequency, which is a subjective matter. The use of the word "helpful" implies that the current timing and frequency are positive, leading the respondent towards a favorable response.




In [97]:
# Let's look at the model history
gpt3_turbo.inspect_history(n=1)





Assess whether a question is biased or not. Keep in mind the following forms of bias:
        (1) Leading questions, which are phrased in a way that suggests a particular answer is more desirable or correct. They tend to have subjective adjectives, or context-laden words that frame the question in a positive or negative light.
        (2) Making assumptions about respondents' behaviors, attitudes, or goals. These questions tend to guess information instead of asking for it.
        (3) Double-barreled questions, which ask about two or more things simultaneously.
        (4) Emotionally loaded language, which has emotionally loaded terms or phrases that imply judgment or assume a particular stance. Words that carry strong positive or negative connotations can influence respondents' emotions and responses.

---

Follow the following format.

Question: The question to classify. The input will be a JSON with the following structure: { "response_format": "open" or "closed", "description

### Create AssessBiasModule

In [4]:
# Create a module from AssessReadability

class AssessBiasModule(dspy.Module):
    
    def __init__(self):

        super().__init__()

        rand_int = random.randint(1, 100)

        # define the reasoning 
        rationale_type = dspy.OutputField(
            prefix="Reasoning: Let's think step by step in order to",
            desc="${determine the bias} We ...",
        )
        
        self.bias = dspy.ChainOfThought(AssessBias, rationale_type=rationale_type, temperature=0.7+0.0001*rand_int)

    def forward(self, question):

        # this needs to return a dict and not a string for Evaluate to work
        return self.bias(question=question)

#### Test AssessBiasModule

In [99]:
bias = AssessBiasModule()

# running the predictor
for test_input in test_inputs:
    # stringify the input
    test_input_str = json.dumps(test_input)
    # test_input_str = test_input["main_text"]
    result = bias(question=test_input_str)
    rationale = result.rationale
    bias_output = result.bias
    print(f"The bias of the question '{test_input["main_text"]}' is {bias_output}.") 
    print(f"Rationale: {rationale}")
    print()
    print()

The bias of the question 'Did the tornado sound like a freight train?' is false.
Rationale: determine the bias. We can see that this question is asking about a specific sound that tornadoes are often compared to. It does not contain leading language, assumptions about the respondent, double-barreled components, or emotionally loaded language.


The bias of the question 'What did the tornado sound like?' is false.
Rationale: determine the bias. We do not see any leading language, assumptions, double-barreled questions, or emotionally loaded language in this question. It simply asks for the description of a tornado sound.


The bias of the question 'Is the current bus timing and frequency helpful to you?' is false.
Rationale: determine the bias. We see that the question is asking for an opinion on whether the bus timing and frequency are helpful, without suggesting a particular answer or assuming the respondent's behavior or attitude. There are no emotionally loaded terms or double-barre

In [100]:
# Let's look at the model history
gpt3_turbo.inspect_history(n=1)





Assess whether a question is biased or not. Keep in mind the following forms of bias:
        (1) Leading questions, which are phrased in a way that suggests a particular answer is more desirable or correct. They tend to have subjective adjectives, or context-laden words that frame the question in a positive or negative light.
        (2) Making assumptions about respondents' behaviors, attitudes, or goals. These questions tend to guess information instead of asking for it.
        (3) Double-barreled questions, which ask about two or more things simultaneously.
        (4) Emotionally loaded language, which has emotionally loaded terms or phrases that imply judgment or assume a particular stance. Words that carry strong positive or negative connotations can influence respondents' emotions and responses.

---

Follow the following format.

Question: The question to classify. The input will be a JSON with the following structure: { "response_format": "open" or "closed", "description

### Optimize AssessBias

In [15]:
filename = "bias_outputs"

# Load the training data
with open(f"generated_questions/{filename}_train.json", 'r') as f:
    train_data = json.load(f)    

# iterate through the data and construct a DSPy Example
trainset_desc = []
for item in train_data:
    # stringify the input
    question_str = json.dumps(item["question"])
    # question_str = item["question"]["main_text"]
    trainset_desc.append(dspy.Example(question=question_str, bias=item["bias"]).with_inputs("question"))   

# iterate through the data and construct a DSPy Example
trainset_no_desc = []
for item in train_data:
    # set the description field to an empty string
    item["question"]["description"] = ""
    # stringify the input
    question_str = json.dumps(item["question"])
    # question_str = item["question"]["main_text"]
    trainset_no_desc.append(dspy.Example(question=question_str, bias=item["bias"]).with_inputs("question"))  

# Load the validation data
with open(f"generated_questions/{filename}_val.json", 'r') as f:
    val_data = json.load(f)

# iterate through the data and construct a DSPy Example
valset = []
for item in val_data:
    # set the description field to an empty string
    item["question"]["description"] = ""
    # stringify the input
    question_str = json.dumps(item["question"])
    # question_str = item["question"]["main_text"]
    valset.append(dspy.Example(question=question_str, bias=item["bias"]).with_inputs("question")) 

print(len(trainset_desc), len(trainset_no_desc), len(valset))
valset[0].question

288 288 96


'{"response_format": "open", "description": "", "main_text": "Are there any city services not currently provided that you believe should be? Please specify.", "response_categories": []}'

In [19]:
trainset_desc[0].question

'{"response_format": "closed", "description": "Assumes respondent perceives a decline in public safety", "main_text": "How significant is the decline in public safety you have experienced in the city?", "response_categories": [{"id": 1, "text": "Very significant"}, {"id": 2, "text": "Significant"}, {"id": 3, "text": "Moderate"}, {"id": 4, "text": "Slight"}, {"id": 5, "text": "No decline"}]}'

In [20]:
trainset_no_desc[0].question

'{"response_format": "closed", "description": "", "main_text": "How significant is the decline in public safety you have experienced in the city?", "response_categories": [{"id": 1, "text": "Very significant"}, {"id": 2, "text": "Significant"}, {"id": 3, "text": "Moderate"}, {"id": 4, "text": "Slight"}, {"id": 5, "text": "No decline"}]}'

In [16]:
# Create a metric
def validate_bias(example, pred, trace=None):

    # make sure rationale is above a certain length
    rationale_length = len(pred.rationale.split(" "))

    # print(pred.rationale)

    if pred.bias.lower() not in ["false", "true"]:
        print(f"Invalid prediction: {pred.bias}")
        # another way of finding the category
        pred_category = ""
        if "false" in pred.bias.lower():
            pred_category = "false"
        elif "true" in pred.bias.lower():
            pred_category = "true"
        else:
            return False
        
        return (example.bias.lower() == pred_category) and (rationale_length > 10)

    return (example.bias.lower() == pred.bias.lower()) and (rationale_length > 10)

In [7]:
non_optimized_bias_program = AssessBiasModule()

In [8]:
# Evaluate the optimized program

evaluate_program = Evaluate(devset=valset, num_threads=1, display_progress=True, display_table=5)

evaluate_program(non_optimized_bias_program, metric=validate_bias)

Average Metric: 85 / 96  (88.5): 100%|██████████| 96/96 [02:24<00:00,  1.50s/it]

Average Metric: 85 / 96  (88.5%)



c:\Users\Cassandra\miniconda3\envs\coalesce\Lib\site-packages\dspy\evaluate\evaluate.py:142: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(truncate_cell)
c:\Users\Cassandra\miniconda3\envs\coalesce\Lib\site-packages\dspy\evaluate\evaluate.py:216: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['✔️ [True]' '✔️ [True]' '✔️ [True]' 'False' '✔️ [True]']' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.loc[:, metric_name] = df[metric_name].apply(lambda x: f'✔️ [{x}]' if x is True else f'{x}')


,question,example_bias,rationale,pred_bias,validate_bias
0,"{""response_format"": ""open"", ""description"": """", ""main_text"": ""Are there any city services not currently provided that you believe should be? Please specify."", ""response_categories"": []}",false,determine the bias. We can see that this question is open-ended and simply asks respondents if there are any city services they believe should be...,false,✔️ [True]
1,"{""response_format"": ""open"", ""description"": """", ""main_text"": ""How frustrated are you with the current level of cleanliness in our city streets and public spaces?"", ""response_categories"": []}",true,determine the bias. We need to examine if the question contains emotionally loaded language that could influence the respondent's answer.,true,✔️ [True]
2,"{""response_format"": ""open"", ""description"": """", ""main_text"": ""Considering the undeniable benefits parks bring to our community's health and happiness, how necessary do you feel it is to...",true,"determine the bias. We see that the question includes emotionally loaded language such as ""undeniable benefits,"" ""health and happiness,"" and ""necessary."" These terms imply a...",true,✔️ [True]
3,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""How satisfied are you with the level of safety measures in public schools?"", ""response_categories"": [{""id"": 1, ""text"": ""Very satisfied""}, {""id"":...",true,determine the bias. We can see that the question is asking about the respondent's satisfaction level with safety measures in public schools. It does not...,false,False
4,"{""response_format"": ""open"", ""description"": """", ""main_text"": ""How transparent do you find the city's budget allocations regarding public services and how inclusive do you find city-sponsored events?"",...",true,determine the bias. We first need to identify if this question falls into any of the bias categories mentioned earlier.,true,✔️ [True]


88.54

In [9]:
# Let's look at the model history
gpt3_turbo.inspect_history(n=1)





Assess whether a question is biased or not. Keep in mind the following forms of bias:
        (1) Leading questions, which are phrased in a way that suggests a particular answer is more desirable or correct. They tend to have subjective adjectives, or context-laden words that frame the question in a positive or negative light.
        (2) Making assumptions about respondents' behaviors, attitudes, or goals. These questions tend to guess information instead of asking for it.
        (3) Double-barreled questions, which ask about two or more things simultaneously.
        (4) Emotionally loaded language, which has emotionally loaded terms or phrases that imply judgment or assume a particular stance. Words that carry strong positive or negative connotations can influence respondents' emotions and responses.

---

Follow the following format.

Question: The question to classify. The input will be a JSON with the following structure: { "response_format": "open" or "closed", "description

In [10]:
# Let's save the optimized programs
non_optimized_bias_program.save('compiled_modules/assess_bias_non_optimized.json')

In [17]:
# set up optimizer 
config = dict(max_bootstrapped_demos=3, max_labeled_demos=5, max_rounds=2, max_errors=5)
# max_bootstrapped_demos: Refers to the maximum number of demonstrations that will be bootstrapped. Bootstrapping in this context likely means generating new training examples based on the predictions of a teacher module or some other process
# max_labeled_demos: Refers to the maximum number of labeled demonstrations (examples) that will be used for training the student module directly

fewshot_optimizer = BootstrapFewShot(metric=validate_bias, **config)
optimized_bias_program_few_shot_no_desc = fewshot_optimizer.compile(AssessBiasModule(), trainset=trainset_no_desc)

  0%|          | 0/288 [00:00<?, ?it/s]

Bootstrapped 3 full traces after 1 examples in round 1.


In [18]:
# Evaluate the optimized program

evaluate_program = Evaluate(devset=valset, num_threads=1, display_progress=True, display_table=5)

evaluate_program(optimized_bias_program_few_shot_no_desc, metric=validate_bias)

Average Metric: 87 / 96  (90.6): 100%|██████████| 96/96 [02:29<00:00,  1.56s/it]

Average Metric: 87 / 96  (90.6%)



c:\Users\Cassandra\miniconda3\envs\coalesce\Lib\site-packages\dspy\evaluate\evaluate.py:142: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(truncate_cell)
c:\Users\Cassandra\miniconda3\envs\coalesce\Lib\site-packages\dspy\evaluate\evaluate.py:216: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['✔️ [True]' '✔️ [True]' '✔️ [True]' 'False' '✔️ [True]']' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.loc[:, metric_name] = df[metric_name].apply(lambda x: f'✔️ [{x}]' if x is True else f'{x}')


,question,example_bias,rationale,pred_bias,validate_bias
0,"{""response_format"": ""open"", ""description"": """", ""main_text"": ""Are there any city services not currently provided that you believe should be? Please specify."", ""response_categories"": []}",false,determine the bias. The question is open-ended and does not make any assumptions or contain leading language. It simply asks respondents if there are any...,false,✔️ [True]
1,"{""response_format"": ""open"", ""description"": """", ""main_text"": ""How frustrated are you with the current level of cleanliness in our city streets and public spaces?"", ""response_categories"": []}",true,determine the bias. The question uses emotionally loaded language by asking about frustration levels in relation to the cleanliness of city streets and public spaces....,true,✔️ [True]
2,"{""response_format"": ""open"", ""description"": """", ""main_text"": ""Considering the undeniable benefits parks bring to our community's health and happiness, how necessary do you feel it is to...",true,"determine the bias. The question uses emotionally loaded language by highlighting the benefits parks bring to the community's health and happiness, which may influence respondents'...",true,✔️ [True]
3,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""How satisfied are you with the level of safety measures in public schools?"", ""response_categories"": [{""id"": 1, ""text"": ""Very satisfied""}, {""id"":...",true,determine the bias. The question asks about the satisfaction level with safety measures in public schools without leading language or assumptions about the respondent's views....,false,False
4,"{""response_format"": ""open"", ""description"": """", ""main_text"": ""How transparent do you find the city's budget allocations regarding public services and how inclusive do you find city-sponsored events?"",...",true,determine the bias. The question is double-barreled as it asks about both the transparency of the city's budget allocations and the inclusivity of city-sponsored events...,true,✔️ [True]


90.62

In [21]:
# Let's look at the model history
gpt3_turbo.inspect_history(n=1)





Assess whether a question is biased or not. Keep in mind the following forms of bias:
        (1) Leading questions, which are phrased in a way that suggests a particular answer is more desirable or correct. They tend to have subjective adjectives, or context-laden words that frame the question in a positive or negative light.
        (2) Making assumptions about respondents' behaviors, attitudes, or goals. These questions tend to guess information instead of asking for it.
        (3) Double-barreled questions, which ask about two or more things simultaneously.
        (4) Emotionally loaded language, which has emotionally loaded terms or phrases that imply judgment or assume a particular stance. Words that carry strong positive or negative connotations can influence respondents' emotions and responses.

---

Follow the following format.

Question: The question to classify. The input will be a JSON with the following structure: { "response_format": "open" or "closed", "description

In [22]:
# set up optimizer 
config = dict(max_bootstrapped_demos=3, max_labeled_demos=5, max_rounds=2, max_errors=5)
# max_bootstrapped_demos: Refers to the maximum number of demonstrations that will be bootstrapped. Bootstrapping in this context likely means generating new training examples based on the predictions of a teacher module or some other process
# max_labeled_demos: Refers to the maximum number of labeled demonstrations (examples) that will be used for training the student module directly

fewshot_optimizer = BootstrapFewShot(metric=validate_bias, **config)
optimized_bias_program_few_shot_desc = fewshot_optimizer.compile(AssessBiasModule(), trainset=trainset_desc)

  0%|          | 0/288 [00:00<?, ?it/s]

Bootstrapped 3 full traces after 1 examples in round 1.


In [23]:
# Evaluate the optimized program

evaluate_program = Evaluate(devset=valset, num_threads=1, display_progress=True, display_table=5)

evaluate_program(optimized_bias_program_few_shot_desc, metric=validate_bias)

Average Metric: 87 / 96  (90.6): 100%|██████████| 96/96 [02:23<00:00,  1.50s/it]

Average Metric: 87 / 96  (90.6%)



c:\Users\Cassandra\miniconda3\envs\coalesce\Lib\site-packages\dspy\evaluate\evaluate.py:142: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(truncate_cell)
c:\Users\Cassandra\miniconda3\envs\coalesce\Lib\site-packages\dspy\evaluate\evaluate.py:216: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['✔️ [True]' '✔️ [True]' '✔️ [True]' 'False' '✔️ [True]']' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.loc[:, metric_name] = df[metric_name].apply(lambda x: f'✔️ [{x}]' if x is True else f'{x}')


,question,example_bias,rationale,pred_bias,validate_bias
0,"{""response_format"": ""open"", ""description"": """", ""main_text"": ""Are there any city services not currently provided that you believe should be? Please specify."", ""response_categories"": []}",false,determine the bias. This question is open-ended and simply asks respondents if there are any city services they believe should be provided but currently are...,false,✔️ [True]
1,"{""response_format"": ""open"", ""description"": """", ""main_text"": ""How frustrated are you with the current level of cleanliness in our city streets and public spaces?"", ""response_categories"": []}",true,determine the bias. This question uses emotionally loaded language by framing the cleanliness of city streets and public spaces in terms of frustration. The word...,true,✔️ [True]
2,"{""response_format"": ""open"", ""description"": """", ""main_text"": ""Considering the undeniable benefits parks bring to our community's health and happiness, how necessary do you feel it is to...",true,"determine the bias. The question uses emotionally loaded language by stating the ""undeniable benefits"" of parks and suggesting that investing more in them is necessary....",true,✔️ [True]
3,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""How satisfied are you with the level of safety measures in public schools?"", ""response_categories"": [{""id"": 1, ""text"": ""Very satisfied""}, {""id"":...",true,determine the bias. This question is straightforward and does not contain leading language or assumptions about the respondent's opinions. It simply asks for a satisfaction...,false,False
4,"{""response_format"": ""open"", ""description"": """", ""main_text"": ""How transparent do you find the city's budget allocations regarding public services and how inclusive do you find city-sponsored events?"",...",true,determine the bias. This question is double-barreled as it asks about two different aspects - transparency in budget allocations and inclusivity in city-sponsored events -...,true,✔️ [True]


90.62

In [24]:
# Let's look at the model history
gpt3_turbo.inspect_history(n=1)





Assess whether a question is biased or not. Keep in mind the following forms of bias:
        (1) Leading questions, which are phrased in a way that suggests a particular answer is more desirable or correct. They tend to have subjective adjectives, or context-laden words that frame the question in a positive or negative light.
        (2) Making assumptions about respondents' behaviors, attitudes, or goals. These questions tend to guess information instead of asking for it.
        (3) Double-barreled questions, which ask about two or more things simultaneously.
        (4) Emotionally loaded language, which has emotionally loaded terms or phrases that imply judgment or assume a particular stance. Words that carry strong positive or negative connotations can influence respondents' emotions and responses.

---

Follow the following format.

Question: The question to classify. The input will be a JSON with the following structure: { "response_format": "open" or "closed", "description

In [25]:
# Let's save the optimized programs
optimized_bias_program_few_shot_no_desc.save('compiled_modules/assess_bias_few_shot_no_desc.json')
optimized_bias_program_few_shot_desc.save('compiled_modules/assess_bias_few_shot_desc.json')

In [26]:
# Optimize the module with BootstrapFewShotWithRandomSearch
# Applies BootstrapFewShot several times with random search over generated demonstrations, and selects the best program
fewshot_optimizer = BootstrapFewShotWithRandomSearch(metric=validate_bias, max_bootstrapped_demos=2, num_candidate_programs=8, num_threads=1)
optimized_bias_program_few_shot_search_no_desc = fewshot_optimizer.compile(student = AssessBiasModule(), trainset=trainset_no_desc, valset=valset)

Going to sample between 1 and 2 traces per predictor.
Will attempt to train 8 candidate sets.


Average Metric: 42 / 51  (82.4):  53%|█████▎    | 51/96 [01:28<01:56,  2.59s/it]

Invalid prediction: false

Question: {"response_format": "open", "description": "", "main_text": "Do you support the dangerous and extreme practices of the opposition party in our government?"

Reasoning: Let's think step by step in order to determine the bias. This question contains emotionally loaded language by describing the practices of the opposition party as "dangerous and extreme." It also makes assumptions about the respondent's stance by implying that supporting these practices is negative.

Bias: true


Average Metric: 81 / 96  (84.4): 100%|██████████| 96/96 [02:37<00:00,  1.64s/it]
c:\Users\Cassandra\miniconda3\envs\coalesce\Lib\site-packages\dspy\evaluate\evaluate.py:142: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(truncate_cell)


Average Metric: 81 / 96  (84.4%)
Score: 84.38 for set: [0]
New best score: 84.38 for seed -3
Scores so far: [84.38]
Best score: 84.38


...

Average Metric: 73 / 96  (76.0%)
Score: 76.04 for set: [16]
Scores so far: [84.38, 93.75, 87.5, 89.58, 79.17, 85.42, 85.42, 89.58, 91.67, 76.04]
Best score: 93.75
Average of max per entry across top 1 scores: 0.9375
Average of max per entry across top 2 scores: 1.0
Average of max per entry across top 3 scores: 1.0
Average of max per entry across top 5 scores: 1.0
Average of max per entry across top 8 scores: 1.0
Average of max per entry across top 9999 scores: 1.0


  1%|          | 2/288 [00:01<04:38,  1.03it/s]


Bootstrapped 2 full traces after 3 examples in round 0.


Average Metric: 85 / 96  (88.5): 100%|██████████| 96/96 [02:04<00:00,  1.30s/it]

Average Metric: 85 / 96  (88.5%)
Score: 88.54 for set: [16]
Scores so far: [84.38, 93.75, 87.5, 89.58, 79.17, 85.42, 85.42, 89.58, 91.67, 76.04, 88.54]
Best score: 93.75
Average of max per entry across top 1 scores: 0.9375
Average of max per entry across top 2 scores: 1.0
Average of max per entry across top 3 scores: 1.0
Average of max per entry across top 5 scores: 1.0
Average of max per entry across top 8 scores: 1.0
Average of max per entry across top 9999 scores: 1.0
11 candidate programs found.


In [27]:
# Evaluate the optimized program

evaluate_program = Evaluate(devset=valset, num_threads=5, display_progress=True, display_table=5)

evaluate_program(optimized_bias_program_few_shot_search_no_desc, metric=validate_bias)

Average Metric: 90 / 96  (93.8): 100%|██████████| 96/96 [00:00<00:00, 1646.61it/s] 

Average Metric: 90 / 96  (93.8%)



c:\Users\Cassandra\miniconda3\envs\coalesce\Lib\site-packages\dspy\evaluate\evaluate.py:216: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['✔️ [True]' '✔️ [True]' '✔️ [True]' 'False' '✔️ [True]']' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.loc[:, metric_name] = df[metric_name].apply(lambda x: f'✔️ [{x}]' if x is True else f'{x}')


,question,example_bias,rationale,pred_bias,validate_bias
0,"{""response_format"": ""open"", ""description"": """", ""main_text"": ""Are there any city services not currently provided that you believe should be? Please specify."", ""response_categories"": []}",false,"determine the bias. We will analyze the question for leading language, assumptions about the respondent's beliefs, and emotionally loaded terms.",false,✔️ [True]
1,"{""response_format"": ""open"", ""description"": """", ""main_text"": ""How frustrated are you with the current level of cleanliness in our city streets and public spaces?"", ""response_categories"": []}",true,determine the bias. We need to analyze if the question contains emotionally loaded language that could influence the respondent's perception.,true,✔️ [True]
2,"{""response_format"": ""open"", ""description"": """", ""main_text"": ""Considering the undeniable benefits parks bring to our community's health and happiness, how necessary do you feel it is to...",true,"determine the bias. We will analyze the question for leading language, assumptions about respondents' attitudes, and emotionally loaded terms.",true,✔️ [True]
3,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""How satisfied are you with the level of safety measures in public schools?"", ""response_categories"": [{""id"": 1, ""text"": ""Very satisfied""}, {""id"":...",true,"determine the bias. We will analyze the question for leading language, assumptions about respondents' behaviors, double-barreled questions, and emotionally loaded language.",false,False
4,"{""response_format"": ""open"", ""description"": """", ""main_text"": ""How transparent do you find the city's budget allocations regarding public services and how inclusive do you find city-sponsored events?"",...",true,"determine the bias. We will analyze if the question contains leading language, assumptions about the respondents' opinions, double-barreled elements, or emotionally loaded language.",true,✔️ [True]


93.75

In [28]:
# Let's look at the model history
gpt3_turbo.inspect_history(n=1)





Assess whether a question is biased or not. Keep in mind the following forms of bias:
        (1) Leading questions, which are phrased in a way that suggests a particular answer is more desirable or correct. They tend to have subjective adjectives, or context-laden words that frame the question in a positive or negative light.
        (2) Making assumptions about respondents' behaviors, attitudes, or goals. These questions tend to guess information instead of asking for it.
        (3) Double-barreled questions, which ask about two or more things simultaneously.
        (4) Emotionally loaded language, which has emotionally loaded terms or phrases that imply judgment or assume a particular stance. Words that carry strong positive or negative connotations can influence respondents' emotions and responses.

---

Follow the following format.

Question: The question to classify. The input will be a JSON with the following structure: { "response_format": "open" or "closed", "description

In [29]:
# Optimize the module with BootstrapFewShotWithRandomSearch
# Applies BootstrapFewShot several times with random search over generated demonstrations, and selects the best program
fewshot_optimizer = BootstrapFewShotWithRandomSearch(metric=validate_bias, max_bootstrapped_demos=2, num_candidate_programs=8, num_threads=5)
optimized_bias_program_few_shot_search_desc = fewshot_optimizer.compile(student = AssessBiasModule(), trainset=trainset_desc, valset=valset)

Going to sample between 1 and 2 traces per predictor.
Will attempt to train 8 candidate sets.


Average Metric: 80 / 96  (83.3): 100%|██████████| 96/96 [00:29<00:00,  3.27it/s]
c:\Users\Cassandra\miniconda3\envs\coalesce\Lib\site-packages\dspy\evaluate\evaluate.py:142: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(truncate_cell)


Average Metric: 80 / 96  (83.3%)
Score: 83.33 for set: [0]
New best score: 83.33 for seed -3
Scores so far: [83.33]
Best score: 83.33


Average Metric: 90 / 96  (93.8): 100%|██████████| 96/96 [00:23<00:00,  4.00it/s]


Average Metric: 90 / 96  (93.8%)
Score: 93.75 for set: [16]
New best score: 93.75 for seed -2
Scores so far: [83.33, 93.75]
Best score: 93.75


...

Average Metric: 76 / 96  (79.2%)
Score: 79.17 for set: [16]
Scores so far: [83.33, 93.75, 89.58, 88.54, 91.67, 95.83, 90.62, 92.71, 91.67, 79.17]
Best score: 95.83
Average of max per entry across top 1 scores: 0.9583333333333334
Average of max per entry across top 2 scores: 1.0
Average of max per entry across top 3 scores: 1.0
Average of max per entry across top 5 scores: 1.0
Average of max per entry across top 8 scores: 1.0
Average of max per entry across top 9999 scores: 1.0


  1%|          | 2/288 [00:01<03:36,  1.32it/s]


Bootstrapped 2 full traces after 3 examples in round 0.


Average Metric: 89 / 96  (92.7): 100%|██████████| 96/96 [00:24<00:00,  3.96it/s] 

Average Metric: 89 / 96  (92.7%)
Score: 92.71 for set: [16]
Scores so far: [83.33, 93.75, 89.58, 88.54, 91.67, 95.83, 90.62, 92.71, 91.67, 79.17, 92.71]
Best score: 95.83
Average of max per entry across top 1 scores: 0.9583333333333334
Average of max per entry across top 2 scores: 1.0
Average of max per entry across top 3 scores: 1.0
Average of max per entry across top 5 scores: 1.0
Average of max per entry across top 8 scores: 1.0
Average of max per entry across top 9999 scores: 1.0
11 candidate programs found.


In [30]:
# Evaluate the optimized program

evaluate_program = Evaluate(devset=valset, num_threads=5, display_progress=True, display_table=5)

evaluate_program(optimized_bias_program_few_shot_search_desc, metric=validate_bias)

Average Metric: 92 / 96  (95.8): 100%|██████████| 96/96 [00:00<00:00, 1714.30it/s] 

Average Metric: 92 / 96  (95.8%)



c:\Users\Cassandra\miniconda3\envs\coalesce\Lib\site-packages\dspy\evaluate\evaluate.py:216: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['✔️ [True]' '✔️ [True]' '✔️ [True]' 'False' '✔️ [True]']' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.loc[:, metric_name] = df[metric_name].apply(lambda x: f'✔️ [{x}]' if x is True else f'{x}')


,question,example_bias,rationale,pred_bias,validate_bias
0,"{""response_format"": ""open"", ""description"": """", ""main_text"": ""Are there any city services not currently provided that you believe should be? Please specify."", ""response_categories"": []}",false,determine the bias. This question is open-ended and does not contain leading language or assumptions about the respondent's opinions. It simply asks for feedback on...,false,✔️ [True]
1,"{""response_format"": ""open"", ""description"": """", ""main_text"": ""How frustrated are you with the current level of cleanliness in our city streets and public spaces?"", ""response_categories"": []}",true,"determine the bias. The question contains emotionally loaded language by using the word ""frustrated"" which implies a negative judgment on the cleanliness of the city...",true,✔️ [True]
2,"{""response_format"": ""open"", ""description"": """", ""main_text"": ""Considering the undeniable benefits parks bring to our community's health and happiness, how necessary do you feel it is to...",true,"determine the bias. The question contains emotionally loaded language by stating that parks bring undeniable benefits to health and happiness, which could influence respondents' opinions.",true,✔️ [True]
3,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""How satisfied are you with the level of safety measures in public schools?"", ""response_categories"": [{""id"": 1, ""text"": ""Very satisfied""}, {""id"":...",true,"determine the bias. The question is straightforward and does not contain leading language, assumptions, double-barreled aspects, or emotionally loaded terms.",false,False
4,"{""response_format"": ""open"", ""description"": """", ""main_text"": ""How transparent do you find the city's budget allocations regarding public services and how inclusive do you find city-sponsored events?"",...",true,"determine the bias. The question is double-barreled as it asks about two different topics in one question, which can lead to confusion and inaccurate responses.",true,✔️ [True]


95.83

In [31]:
# Let's look at the model history
gpt3_turbo.inspect_history(n=1)





Assess whether a question is biased or not. Keep in mind the following forms of bias:
        (1) Leading questions, which are phrased in a way that suggests a particular answer is more desirable or correct. They tend to have subjective adjectives, or context-laden words that frame the question in a positive or negative light.
        (2) Making assumptions about respondents' behaviors, attitudes, or goals. These questions tend to guess information instead of asking for it.
        (3) Double-barreled questions, which ask about two or more things simultaneously.
        (4) Emotionally loaded language, which has emotionally loaded terms or phrases that imply judgment or assume a particular stance. Words that carry strong positive or negative connotations can influence respondents' emotions and responses.

---

Follow the following format.

Question: The question to classify. The input will be a JSON with the following structure: { "response_format": "open" or "closed", "description

In [32]:
# Let's save the optimized programs
optimized_bias_program_few_shot_search_no_desc.save('compiled_modules/assess_bias_few_shot_search_no_desc.json')
optimized_bias_program_few_shot_search_desc.save('compiled_modules/assess_bias_few_shot_search_desc.json')

### Create RewriteQuestion Signature

In [33]:
# Create another class-based DSPy Signature (re-write a question based on rationale)
    
input_description = """The question to classify. The input will be a JSON with the following structure:
    {
        "response_format": "open" or "closed",
        "description": string,
        "main_text": string,
        "response_categories": empty list or list of JSONs with an "id" and "text" field
    }"""

output_description = """The re-written question. The output will be a JSON with the following structure:
    {
        "response_format": "open" or "closed",
        "description": string,
        "main_text": string,
        "response_categories": empty list or list of JSONs with an "id" and "text" field
    }"""
    
class RewriteQuestion(dspy.Signature):
    """Rewrite a question to address the issues identified in the input_rationale."""

    question = dspy.InputField(desc=input_description)
    input_rationale = dspy.InputField(desc="The rationale with potential issues in a question. It may contain a mix of positive and negative feedback.")
    rewritten_question = dspy.OutputField(desc=output_description)

### Create AssessBiasAndRewrite

In [ ]:
# Create a Module with optimized_readability_program2 and RewriteQuestion

class AssessBiasAndRewrite(dspy.Module):
    
    def __init__(self, optimized_bias_program):

        super().__init__()

        self.bias = optimized_bias_program

        self.rewrite = dspy.ChainOfThought(RewriteQuestion)

    def forward(self, question):

        result = self.bias(question=question)

        bias_score = result.bias
        input_rationale = result.rationale

        assert bias_score in ["true", "false"]

        if bias_score == "true":
            return self.rewrite(question=question, input_rationale=input_rationale).rewritten_question, input_rationale, bias_score
        else:
            return question, input_rationale, bias_score

In [35]:
test_inputs = [
    {
        "question": {
            "response_format": "closed",
            "description": "",
            "main_text": "Did the tornado sound like a freight train?",
            "response_categories": [
                {"id": 0, "text": "Yes"},
                {"id": 1, "text": "No"}
            ]
        },
        "bias": "true"
    },
    {
        "question": {
            "response_format": "closed",
            "description": "",
            "main_text": "Did you try to put out the fire?",
            "response_categories": [
                {"id": 0, "text": "Yes"},
                {"id": 1, "text": "No"}
            ]
        },
        "bias": "true"
    },
    {
        "question": {
            "response_format": "open",
            "description": "",
            "main_text": "What did the tornado sound like?",
            "response_categories": []
        },
        "bias": "false"
    },
    {
        "question": {
            "response_format": "open",
            "description": "",
            "main_text": "What did you do when you saw the fire?",
            "response_categories": []
        },
        "bias": "false"
    },
    {
        "question": {
            "response_format": "open",
            "description": "",
            "main_text": "How do you like our amazing city?",
            "response_categories": []
        },
        "bias": "true"
    },
    {
        "question": {
            "response_format": "closed",
            "description": "",
            "main_text": "More and more people have come to accept using a tax preparer to reduce one’s tax burden as beneficial. Do you feel that using a tax preparer to reduce your tax burden is beneficial?",
            "response_categories": [
                {"id": 0, "text": "Yes"},
                {"id": 1, "text": "No"}
            ]
        },
        "bias": "true"
    },
    {
        "question": {
            "response_format": "closed",
            "description": "",
            "main_text": "Should government provide more help to the poor? In our capitalist economy, should the health care be universal?",
            "response_categories": [
                {"id": 0, "text": "Yes"},
                {"id": 1, "text": "No"}
            ]
        },
        "bias": "true"
    },
    {
        "question": {
            "response_format": "open",
            "description": "",
            "main_text": "How can the employees be made efficient?",
            "response_categories": []
        },
        "bias": "true"
    },
    {
        "question": {
            "response_format": "open",
            "description": "",
            "main_text": "What measures may advance the employees’ performance?",
            "response_categories": []
        },
        "bias": "false"
    },
    {
        "question": {
            "response_format": "closed",
            "description": "",
            "main_text": "Do you believe the US should immediately withdraw troops from the failed war in Iraq?",
            "response_categories": [
                {"id": 0, "text": "Yes"},
                {"id": 1, "text": "No"}
            ]
        },
        "bias": "true"
    },
    {
        "question": {
            "response_format": "closed",
            "description": "",
            "main_text": "Do you support or oppose the death tax?",
            "response_categories": [
                {"id": 0, "text": "Support"},
                {"id": 1, "text": "Oppose"}
            ]
        },
        "bias": "true"
    },
    {
        "question": {
            "response_format": "closed",
            "description": "",
            "main_text": "Do you support or oppose President Bush’s plan to require standardized testing of all public school students?",
            "response_categories": [
                {"id": 0, "text": "Support"},
                {"id": 1, "text": "Oppose"}
            ]
        },
        "bias": "true"
    },
    {
        "question": {
            "response_format": "closed",
            "description": "",
            "main_text": "Do you support or oppose higher taxes so that children can have a better start in life?",
            "response_categories": [
                {"id": 0, "text": "Support"},
                {"id": 1, "text": "Oppose"}
            ]
        },
        "bias": "true"
    },
    {
        "question": {
            "response_format": "open",
            "description": "",
            "main_text": "Some people support very low levels of government involvement in the economy, while others support very high levels of government involvement. How much government involvement do you support?",
            "response_categories": []
        },
        "bias": "false"
    },
    {
        "question": {
            "response_format": "closed",
            "description": "",
            "main_text": "Do you favor increasing the tax rate in the top bracket?",
            "response_categories": [
                {"id": 0, "text": "Support"},
                {"id": 1, "text": "Oppose"}
            ]
        },
        "bias": "true"
    },
    {
        "question": {
            "response_format": "closed",
            "description": "",
            "main_text": "Do you favor or oppose increasing the tax rate in the top bracket?",
            "response_categories": [
                {"id": 0, "text": "Support"},
                {"id": 1, "text": "Oppose"}
            ]
        },
        "bias": "false"
    },
    {
        "question": {
            "response_format": "open",
            "description": "",
            "main_text": "Is the current bus timing and frequency helpful to you?",
            "response_categories": []
        },
        "bias": "true"
    },
    {
        "question": {
            "response_format": "closed",
            "description": "",
            "main_text": "Do you support or oppose increasing the estate tax and the personal income tax in the top bracket?",
            "response_categories": [
                {"id": 0, "text": "Support"},
                {"id": 1, "text": "Oppose"}
            ]
        },
        "bias": "true"
    },
    {
        "question": {
            "response_format": "closed",
            "description": "",
            "main_text": "Do you think the president should lower taxes and spending?",
            "response_categories": [
                {"id": 0, "text": "Yes"},
                {"id": 1, "text": "No"}
            ]
        },
        "bias": "true"
    },
    {
        "question": {
            "response_format": "closed",
            "description": "",
            "main_text": "Did you experience sore throat and fever during your cough and cold?",
            "response_categories": [
                {"id": 0, "text": "Yes"},
                {"id": 1, "text": "No"}
            ]
        },
        "bias": "true"
    },
]

In [40]:
# Test out AssessReadabilityAndRewrite

# rand_int = random.randint(1, 100)

assess_bias = AssessBiasModule()
assess_bias.load('compiled_modules/assess_bias_few_shot_search_desc.json')

# # create the module object
# assess_bias_and_rewrite = AssessBiasAndRewrite(assess_bias)

num_correct = 0

# running the predictor
for test_input in test_inputs:
    # stringify the input
    test_input_str = json.dumps(test_input["question"])
    expected_bias = test_input["bias"]
    result = assess_bias(question=test_input_str)
    rationale = result.rationale
    bias_output = result.bias
    if expected_bias == bias_output:
        num_correct += 1
    print(f"The bias of the question '{test_input["question"]["main_text"]}' is {bias_output}.") 
    print(f"Rationale: {rationale}")
    print()
    print()

    # rewritten_question, rationale, bias_score = assess_bias_and_rewrite(question=test_input_str)
    # if expected_bias == bias_score:
    #     num_correct += 1
    # print(f"The expcted bias is {expected_bias}. The actual bias is {bias_score}.")
    # print(f"The original question is: {test_input_str}.")
    # print(f"The re-written question is: {rewritten_question}.")
    # print(f"The rationale is: {rationale}.")
    # print()
    # print()

print(f"Number of correct predictions: {num_correct} out of {len(test_inputs)}. {num_correct/len(test_inputs)*100}% accuracy.")

The bias of the question 'Did the tornado sound like a freight train?' is false.
Rationale: determine the bias. We can see that the question is straightforward and does not contain any leading language or assumptions.


The bias of the question 'Did you try to put out the fire?' is false.
Rationale: determine the bias. This question is straightforward and does not contain any leading language, assumptions, double-barreled questions, or emotionally loaded terms.


The bias of the question 'What did the tornado sound like?' is false.
Rationale: determine the bias. This question is asking for a sensory description of a tornado, so it does not exhibit any of the bias types mentioned.


The bias of the question 'What did you do when you saw the fire?' is false.
Rationale: determine the bias. This question is straightforward and does not contain leading language, assumptions, double-barreled elements, or emotionally loaded terms.


The bias of the question 'How do you like our amazing city?'